# 14 — Pipeline de Producción

## Introducción

**Objetivo.** Empaquetar y documentar el modelo final para integrarlo en PREDIA.

**Fundamento teórico.** Un pipeline serializado (preprocesamiento + modelo) garantiza consistencia train/serving.

**Ventajas.** Reproducible, versionado y fácil de servir; evita 'training-serving skew'.

**Limitaciones.** Debe re-entrenarse periódicamente; monitorear deriva de datos.

**Casos de uso.** Despliegue del modelo en la API de PREDIA.


In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../src'))
import warnings; warnings.simplefilter('ignore')
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
from predia_ml import config, data, evaluate, plots
pd.set_option('display.max_columns', 60)


In [2]:
card = json.load(open(config.EXPORTS_DIR/'model_card.json'))
print(json.dumps(card, indent=2, ensure_ascii=False))

{
  "model_name": "logistic_regression",
  "framing": "screening (sin laboratorios diagnósticos — honesto)",
  "features": [
    "gender",
    "ethnicity",
    "education_level",
    "income_level",
    "employment_status",
    "smoking_status",
    "family_history_diabetes",
    "hypertension_history",
    "cardiovascular_history",
    "age",
    "alcohol_consumption_per_week",
    "physical_activity_minutes_per_week",
    "diet_score",
    "sleep_hours_per_day",
    "screen_time_hours_per_day",
    "bmi",
    "waist_to_hip_ratio",
    "systolic_bp",
    "diastolic_bp",
    "heart_rate",
    "cholesterol_total",
    "hdl_cholesterol",
    "ldl_cholesterol",
    "triglycerides"
  ],
  "target": "diagnosed_diabetes",
  "test_metrics": {
    "accuracy": 0.6044,
    "balanced_accuracy": 0.613625,
    "precision": 0.7144355853965589,
    "recall_sensitivity": 0.5675,
    "specificity": 0.65975,
    "f1": 0.6325469069292217,
    "mcc": 0.22290285680445154,
    "tn": 5278,
    "fp": 2722,
  

### Carga del pipeline y predicción de ejemplo (extremo a extremo)

In [3]:
est = joblib.load(config.EXPORTS_DIR/'predia_diabetes_model.joblib')
df = data.load_raw()
from predia_ml import preprocess
X, y = preprocess.make_xy(df, 'screening')
ejemplo = X.iloc[[0, 1, 2]]
proba = est.predict_proba(ejemplo)[:,1]
pd.DataFrame({'pred': est.predict(ejemplo), 'P(diabetes)': proba.round(4), 'real': y.iloc[:3].values})

,pred,P(diabetes),real
0,1,0.5246,1
1,0,0.3881,0
2,1,0.7086,1


### Integración con PREDIA
El pipeline recibe el **dataframe crudo de features de cribado** (sin labs diagnósticos) y devuelve probabilidad + clase. La migración del endpoint `/api/predicciones/nueva` al nuevo contrato de features se documenta en `reports/model_report.md` y `comparisons/current_vs_new_models.md`.